In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

COLUMN_MAP = {
    "x1": "limit_bal",
    "x2": "sex",
    "x3": "education",
    "x4": "marriage",
    "x5": "age",
    "x6": "pay_0", 
    "x7": "pay_2", 
    "x8": "pay_3", 
    "x9": "pay_4",
    "x10": "pay_5", 
    "x11": "pay_6",
    "x12": "bill_amt1", 
    "x13": "bill_amt2", 
    "x14": "bill_amt3",
    "x15": "bill_amt4", 
    "x16": "bill_amt5", 
    "x17": "bill_amt6",
    "x18": "pay_amt1", 
    "x19": "pay_amt2", 
    "x20": "pay_amt3",
    "x21": "pay_amt4", 
    "x22": "pay_amt5", 
    "x23": "pay_amt6",
}

raw = fetch_openml(data_id=42477, as_frame=True, parser="auto")
X = raw.data.rename(columns=COLUMN_MAP)
y = raw.target.astype(int)

CLASS_NAMES = ["No Default", "Default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)}\n")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

print("Test samples available:", X_test_scaled.shape[0])



Accuracy: 0.6796666666666666

              precision    recall  f1-score   support

  No Default       0.87      0.70      0.77      4673
     Default       0.37      0.62      0.46      1327

    accuracy                           0.68      6000
   macro avg       0.62      0.66      0.62      6000
weighted avg       0.76      0.68      0.70      6000

Test samples available: 6000


In [47]:
feature_names = X_train_scaled.columns.tolist()

shap_explainer = shap.LinearExplainer(model, X_train_scaled)
shap_exp = shap_explainer(X_test_scaled) 

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=feature_names,
    class_names=CLASS_NAMES,
    mode="classification",
    random_state=42,                                
)

def shap_top(i, k = 5):
    vals = shap_exp.values[i]
    order = np.argsort(np.abs(vals))[::-1][:k]
    return [feature_names[j] for j in order]

def lime_top(i, k = 5):
    exp = lime_explainer.explain_instance(X_test_scaled.iloc[i].values, model.predict_proba, num_features=k)
    pairs = sorted(exp.as_map()[1], key=lambda t: abs(t[1]), reverse=True)[:k]
    return [feature_names[idx] for idx, w in pairs]


sample_idx = [0,1,2,3,4]
for i in sample_idx:
    prediction = CLASS_NAMES[model.predict(X_test_scaled.iloc[[i]])[0]]
    print(f"\nCustomer {i} is predicted to be {prediction}")
    print("SHAP Top-5 Features:", shap_top(i))
    print("LIME Top-5 Features:", lime_top(i))



Background dataset has 24000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=24000 when initializing the masker.
/Users/beratmertkayacan/Desktop/TrustAI/SHAP-LIME_Task1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(



Customer 0 is predicted to be No Default
SHAP Top-5 Features: ['pay_0', 'pay_amt1', 'bill_amt1', 'limit_bal', 'sex']
LIME Top-5 Features: ['pay_0', 'pay_amt1', 'bill_amt1', 'limit_bal', 'pay_3']

Customer 1 is predicted to be No Default
SHAP Top-5 Features: ['pay_0', 'pay_3', 'bill_amt1', 'education', 'pay_amt1']


/Users/beratmertkayacan/Desktop/TrustAI/SHAP-LIME_Task1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/beratmertkayacan/Desktop/TrustAI/SHAP-LIME_Task1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


LIME Top-5 Features: ['pay_0', 'bill_amt1', 'pay_3', 'pay_2', 'education']

Customer 2 is predicted to be No Default
SHAP Top-5 Features: ['limit_bal', 'age', 'sex', 'pay_amt1', 'marriage']
LIME Top-5 Features: ['limit_bal', 'age', 'marriage', 'pay_amt1', 'pay_amt2']

Customer 3 is predicted to be No Default
SHAP Top-5 Features: ['bill_amt1', 'bill_amt2', 'bill_amt5', 'bill_amt4', 'bill_amt6']


/Users/beratmertkayacan/Desktop/TrustAI/SHAP-LIME_Task1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


LIME Top-5 Features: ['bill_amt1', 'pay_amt1', 'pay_amt2', 'limit_bal', 'bill_amt2']

Customer 4 is predicted to be No Default
SHAP Top-5 Features: ['pay_0', 'limit_bal', 'pay_2', 'bill_amt1', 'pay_3']
LIME Top-5 Features: ['pay_0', 'pay_amt1', 'pay_amt2', 'limit_bal', 'bill_amt1']


/Users/beratmertkayacan/Desktop/TrustAI/SHAP-LIME_Task1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
